# IFC objects: read, understand, change, check
A first COMPAS IFC lesson for bachelor students. No Python experience required. Allow about 45–60 minutes.

**Your task:** explore a building, select one object, change its name in a new IFC file, and check that the change survived saving.

By the end you can:
- Explain the difference between an IFC object and its visible shape.
- Select objects by type and read their identity and properties.
- Make a small edit, save it, and verify the saved result.
- Export a selected group of IFC objects while keeping their identity.

**On ifc.io:** follow the explanations and inspect the worked example. Running real COMPAS IFC code requires the Python notebook service linked by your course. The sample IFC is included with the lesson.

## 0. Why IFC and COMPAS?
IFC (Industry Foundation Classes) is a format for exchanging building information. A wall can have a shape, a name, a class, property sets, and relationships to a storey or other objects.

**Think of an IFC object as an object card linked to a shape.** `GlobalId` identifies the card; `Name` is a readable label; `IfcWall` is its class; a property set groups named values. Some fields may be absent. Two identical-looking shapes can carry different information.

`compas_ifc` gives Python access to building information and connects it to COMPAS geometry. Code helps repeat the same selection or change on many objects. We start with one small change so we can understand and check it.

Our journey: **IFC file → model in memory → selection → edit → new IFC file → reopen and check**. An optional final step exports a smaller IFC selection.

**Predict:** Would changing a wall's colour in a viewer necessarily change its fire-rating property? Explain why.

## 1. Get ready
This lesson includes `sample_building.ifc`, a small wall and window model from the COMPAS IFC repository. Use it for your first run. Later, your instructor may provide a course model.

- The sample IFC and notebook should be in the same folder in the hosted notebook service.
- Open the hosted notebook from the course website and choose its Python kernel. You do not need to install Python on your own computer.
- Run the installation cell below once. `%pip` installs into the current notebook kernel, so it is simpler and safer than copying a command into another terminal.
- Click a code cell and press **Shift + Enter** to run it. Run from top to bottom. Text cells explain; code cells perform operations.

**Instructor:** the website links to a public Binder notebook with this sample and its Python dependencies. Test the Binder launch before class. Older tutorials using `compas_ifc.model.Model` describe a different API.

**Tiny Python guide:** `=` stores a value under a name; quotes enclose text; `#` starts a comment; `object.name` accesses a field; `function(...)` calls an operation. Indentation groups instructions. Initially change only settings marked `CHANGE`.

Cells remember variables until the kernel restarts. Editing code does nothing until you run it. If results seem inconsistent, restart the kernel and run the lesson from the top. Binder sessions are temporary: download your edited IFC before closing the browser tab.

### Install COMPAS IFC
Run this cell once before the rest of the lesson. It installs `compas_ifc` and its required Python dependencies into this notebook's kernel. Installation can take a few minutes.

If Jupyter asks you to restart the kernel, use **Kernel → Restart Kernel**, then run the notebook from this point again. On a course server that already has COMPAS IFC, this cell simply checks that the dependency is present.

In [ ]:
%pip install compas_ifc

In [ ]:
from pathlib import Path
import sys
from importlib.metadata import version, PackageNotFoundError

print("Python executable:", sys.executable)
print("Python version:", sys.version.split()[0])
for package in ["compas", "compas_ifc"]:
    try:
        print(package, version(package))
    except PackageNotFoundError:
        print(package, "is missing: choose the course kernel or ask your instructor.")

### Choose your file
Change the filename below. A path is a file's location; `Path` helps Python work with it. Relative paths start from the working folder printed below. Keep the quotes.

**Expected:** the full path to an existing IFC file. Fix a missing file error before continuing.

In [ ]:
IFC_PATH = Path("sample_building.ifc")  # CHANGE only when using another model
MODEL_SOURCE = "COMPAS IFC repository: data/wall-with-opening-and-window.ifc"

print("Working folder:", Path.cwd())
if not IFC_PATH.is_file():
    raise FileNotFoundError(f"Cannot find {IFC_PATH.resolve()}. Change IFC_PATH or move the file here.")
print("Input:", IFC_PATH.resolve())
print("Source:", MODEL_SOURCE)

## 2. Open the building information model
`model` is our Python handle to the IFC information. Opening reads the file into memory; it does not edit the disk file.

We disable geometry loading and placement rectification because our first questions concern data. Geometry export later uses a separate operation that loads what it needs.

**Expected:** the model opens. Large files may take longer. If the import fails, check the kernel and package version.

In [ ]:
from compas_ifc.bim import BuildingInformationModel

model = BuildingInformationModel(
    str(IFC_PATH), load_geometries=False, rectify_placements=False
)
print("Model opened. The source file has not been saved over.")

## 3. Find objects by type
**Predict:** which classes should your building contain? `IfcWall` also matches subclasses such as `IfcWallStandardCase`.

`for` repeats the indented instruction; `len(...)` counts a list. Zero is a useful finding, not a coding failure. Classification may differ from what a shape suggests.

**With the included sample:** expect one wall, one window, no doors, and no slabs. The wall is stored as `IfcWallStandardCase`, a subclass matched by the `IfcWall` query.

In [ ]:
for ifc_type in ["IfcWall", "IfcWindow", "IfcDoor", "IfcSlab"]:
    found = model.get_elements_by_type(ifc_type)
    print(ifc_type, ":", len(found))

### Select one class
Choose a class with a nonzero count. `elements` is a list; `[:5]` previews its first five items. Selection changes nothing.

If you change the class, rerun this cell and the following cells to rebuild the selection and outputs.

In [ ]:
ELEMENT_TYPE = "IfcWall"  # CHANGE: IfcWall, IfcWindow, IfcDoor, or IfcSlab
elements = model.get_elements_by_type(ELEMENT_TYPE)
print("Matching objects:", len(elements))
for index, element in enumerate(elements[:5]):
    print(index, element.ifc_type, element.name, element.global_id)

### Pick an object and read its card
Lists count from **zero**: `elements[0]` is the first item. Choose an index printed above. The index is temporary; `GlobalId` is the IFC identifier we use to find the object again. Names can be missing or repeated.

**With the included sample:** the wall is named `Wall for Test Example` and has GlobalId `3ZYW59sxj8lei475l7EhLU`. Its list index is `0`. Record the identifier. Why would matching by name alone be unreliable?

In [ ]:
OBJECT_INDEX = 0  # CHANGE: an index from the preview
if not elements:
    raise ValueError("Choose a type with a nonzero count and rerun the selection.")
if not 0 <= OBJECT_INDEX < len(elements):
    raise ValueError("Choose an index within the matching list.")
selected = elements[OBJECT_INDEX]
SELECTED_ID = selected.global_id
print("Type:", selected.ifc_type)
print("Name:", selected.name)
print("GlobalId:", SELECTED_ID)

## 4. Read properties and location
A property set groups values authored in the model. A dictionary stores named values. **Missing information is not zero or false.** An authored value is not an independently verified result.

The element is a COMPAS-facing wrapper. In this guided cell, `_ifc_entity` reaches its IFC-specific wrapper for properties and parent relationships. The underscore marks an implementation detail that can change between releases; we isolate that access here.

**With the included sample:** the wall has `Pset_WallCommon`, including `IsExternal: true` and `ThermalTransmittance: 0.24`; its parent is `Default Building Storey`. These are authored model values. The property output alone does not tell you the unit or prove physical performance.

In [ ]:
import json

ifc_object = selected._ifc_entity
property_sets = ifc_object.property_sets or {}
print(json.dumps(property_sets, indent=2, default=str) if property_sets else "No property sets reported.")
parent = ifc_object.parent
print("Parent:", getattr(parent, "Name", None) if parent is not None else "No parent reported")

### Check your understanding
- Find one property and describe its meaning. If none are present, record that finding.
- Is its unit declared? What would you check before comparing files?
- Does a name containing “concrete” prove the object's material?

Before comparing numerical properties, check units. Do not assume `2` means two metres. The optional CLI section below reports file units.

## 5. Change one object's name
We change a **teaching label**, not an engineering property. Reloading a fresh working model makes this step repeatable from the original. We find the same object by GlobalId.

Assigning `.name` changes the working model in memory. No output IFC exists until we save.

**Predict:** should changing only the name change the object's class, identifier, or shape? With the included sample, the before name is `Wall for Test Example`; the after name is `Teaching example - renamed object`.

In [ ]:
NEW_NAME = "Teaching example - renamed object"  # CHANGE
working_model = BuildingInformationModel(
    str(IFC_PATH), load_geometries=False, rectify_placements=False
)
working_object = working_model.get_element_by_global_id(SELECTED_ID)
if working_object is None:
    raise ValueError("Selected GlobalId not found. Rerun the selection on your input file.")
old_name = working_object.name
working_object.name = NEW_NAME
print("Before:", old_name)
print("After:", working_object.name)
print("Identifier unchanged:", working_object.global_id == SELECTED_ID)

## 6. Save a new IFC and reopen it
Saving writes our working model to a **new file**. The guard prevents replacing an existing output. For another run, choose a new output name or deliberately remove your previous exercise output in your file manager.

Reopening matters: a correct value in memory does not prove it was saved. `assert` stops if a comparison fails.

**Expected with the sample:** reopening finds `Teaching example - renamed object` under the same GlobalId `3ZYW59sxj8lei475l7EhLU`, still an `IfcWallStandardCase`. This checks this edit; it does not validate every relationship or geometry in the file.

In [ ]:
OUTPUT_DIR = Path("exports")
OUTPUT_DIR.mkdir(exist_ok=True)
EDITED_IFC_PATH = OUTPUT_DIR / "renamed_building.ifc"  # CHANGE for another run
if EDITED_IFC_PATH.resolve() == IFC_PATH.resolve() or EDITED_IFC_PATH.exists():
    raise FileExistsError("Choose a new EDITED_IFC_PATH; existing IFC files are protected.")
working_model.save(str(EDITED_IFC_PATH))
reopened = BuildingInformationModel(
    str(EDITED_IFC_PATH), load_geometries=False, rectify_placements=False
)
checked = reopened.get_element_by_global_id(SELECTED_ID)
assert checked is not None, "Object missing from saved file."
assert checked.name == NEW_NAME, "Saved name differs from requested name."
assert checked.ifc_type == selected.ifc_type, "IFC class changed."
print("Checked name:", checked.name)
print("Checked GlobalId:", checked.global_id)
print("New IFC:", EDITED_IFC_PATH.resolve())

### Record your change
This JSON exercise log helps another student reproduce your work. It is not a complete model comparison or an engineering approval.

In [ ]:
record = {
    "input": str(IFC_PATH.resolve()), "source": MODEL_SOURCE,
    "output": str(EDITED_IFC_PATH.resolve()), "global_id": SELECTED_ID,
    "old_name": old_name, "new_name": checked.name,
    "python": sys.version.split()[0],
    "packages": {p: version(p) for p in ["compas", "compas_ifc"]},
    "check": "reopened name, GlobalId lookup, and IFC class",
}
RECORD_PATH = EDITED_IFC_PATH.with_suffix(".lesson.json")
RECORD_PATH.write_text(json.dumps(record, indent=2), encoding="utf-8")
print("Exercise record:", RECORD_PATH.resolve())

## 7. Optional: inspect using the CLI
The command-line interface is another way to use the library. Read the helper without memorising it: `subprocess` runs another program with the notebook's Python; `--json` asks for structured output; `json.loads` converts that into Python values.

CLI commands depend on the release. For an unknown command, ask your instructor to compare `python -m compas_ifc --help` with the lesson.

**Expected:** file schema, units, and counts; then a spatial hierarchy. Compare the hierarchy with the parent you read earlier.

In [ ]:
import subprocess

def run_ifc(*arguments, json_output=True):
    command = [sys.executable, "-m", "compas_ifc", *map(str, arguments)]
    if json_output:
        command.append("--json")
    result = subprocess.run(command, capture_output=True, text=True)
    if result.returncode != 0:
        raise RuntimeError(f"COMPAS IFC command failed:\n{result.stderr or result.stdout}")
    if not json_output:
        return result.stdout
    try:
        return json.loads(result.stdout)
    except json.JSONDecodeError as error:
        raise RuntimeError(f"Expected JSON. Check the CLI version/output:\n{result.stdout}") from error

print(json.dumps(run_ifc("info", IFC_PATH), indent=2))

In [ ]:
print(json.dumps(run_ifc("tree", IFC_PATH, "--depth", 3), indent=2))

### Export the selected class as IFC
This exports the selected class from the **original** file, so your name edit is absent. Use `EDITED_IFC_PATH` as the input if you want that edit included.

By default, `export-ifc` retains the selection's spatial ancestors and linked information. A subset is not the whole building. Reopen it and compare selected GlobalIds; additional spatial objects are expected.

**Expected:** matching selected-class identifiers. This is a focused identity check, not complete validation of geometry, materials, or relationships.

In [ ]:
SUBSET_PATH = OUTPUT_DIR / f"{ELEMENT_TYPE.lower()}_subset.ifc"
if SUBSET_PATH.exists() or SUBSET_PATH.resolve() == IFC_PATH.resolve():
    raise FileExistsError("Choose a new SUBSET_PATH before exporting again.")
print(run_ifc("export-ifc", IFC_PATH, "--type", ELEMENT_TYPE, "--out", SUBSET_PATH, json_output=False))
subset = BuildingInformationModel(str(SUBSET_PATH), load_geometries=False, rectify_placements=False)
subset_elements = subset.get_elements_by_type(ELEMENT_TYPE)
assert {e.global_id for e in subset_elements} == {e.global_id for e in elements}
print("Checked selected-class IDs:", len(subset_elements))

## 8. Practice and hand in
- Select another class. Record its count and one GlobalId. Explain why class matters even when shapes look alike.
- Rename another object, save to a different filename, and reopen it. Hand in the exercise record and new IFC if sharing is permitted.
- Explain the difference between selecting, editing in memory, and saving. Which makes a persistent change?
- Optional: export windows instead of walls and check which GlobalId appears in the smaller IFC.

**Self-check:** a name edit keeps identifier and class; selection changes nothing; saving makes the edit persistent; reopening checks persistence; a subset contains selected objects plus needed context.

In Binder's left file browser, open `exports`, right-click `renamed_building.ifc` and `renamed_building.lesson.json`, and choose **Download** for each before closing the session. Include the input source, package versions, GlobalId, before/after name, and reopened-file check. Report only checks you performed.

## Help when a cell fails
- **ModuleNotFoundError:** choose the course kernel; compare its Python executable with the one printed above.
- **FileNotFoundError:** compare the working folder and full path with your file's location.
- **NameError:** an earlier cell has not run. Restart and run the core lesson in order.
- **No matching objects:** choose a nonzero class from the counts; classification varies between files.
- **FileExistsError:** choose a new output filename; previous IFC outputs are protected.
- **Unknown CLI command or attribute:** record the package version and ask the instructor to check API compatibility.
- **Binder launch fails:** the public course repository or its image may be unavailable. Tell your instructor the message shown by Binder.

When asking for help, share the cell heading, versions, and final error message. Avoid sharing a restricted model.

## References and instructor notes
Revised using COMPAS Brain MCP's `compas_ifc` skill, cookbook, and workflows. Retrieved guidance is source information, not evidence of execution on a model.

- [COMPAS IFC source and examples](https://github.com/compas-dev/compas_ifc)
- [BuildingInformationModel API](https://compas.dev/compas_ifc/2.1.0/api/generated/compas_ifc.bim.BuildingInformationModel.html)
- [Command-line interface](https://compas.dev/compas_ifc/latest/cli.html)

The lesson, including the optional IFC subset export, was run on the included sample using Python 3.12.13, COMPAS 2.15.1, and `compas_ifc` 2.1.0 on Windows. The public Binder launch still needs a live check after the course repository is public.

Suggested timing: 10 minutes on concepts and controls, 15 on selection and properties, 20 on the checked edit, and 10 on discussion or the optional subset export. Students use the course website and Binder in their browser.